> 在线讲义：[第 10 章 · 慢想快做：分层系统](https://wikieden.github.io/vla-from-scratch/lessons/10-hierarchy.html) · 本书《VLA 从零到 π0》

# 第 10 章 · 慢想快做：分层系统

配套讲义：仓库里 `vla-deep-dive/notes/lessons/10-hierarchy.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `high_level_calls(8, 1.0, [])` | 8 次 |
| 第 1 步 | `high_level_calls(8, 1.0, [4.3])`、`low_level_actions(8, 50)` | 9 次（4.3 秒插话后改从 4.3 起计时）；400 个动作 |
| 第 1 步 | `SYSTEMS` | GR00T N1：10 Hz / 120 Hz、16 步 63.9 ms |

任务和插话时刻是编的；节奏按 Hi Robot、GR00T N1 原文。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
SYSTEMS = {"GR00T N1": {"高层 Hz": 10, "低层 Hz": 120, "16 步 chunk 用时 ms": 63.9}}

**函数 `high_level_calls`**：高层想几次：从 0 秒起每 period 一次；插话立刻想一次，之后从插话时刻重新计时。

In [ ]:
def high_level_calls(task_s, period_s, interrupts):
    """高层想几次：从 0 秒起每 period 一次；插话立刻想一次，之后从插话时刻重新计时。"""
    times, t, pending = [], 0.0, sorted(interrupts)
    while t < task_s - 1e-9:
        times.append(round(t, 3))
        nxt = t + period_s
        if pending and t < pending[0] < nxt - 1e-9:
            nxt = pending.pop(0)  # 用户插话：不等满一个周期
        t = nxt
    return times

**函数 `low_level_actions`**：低层一共出多少个动作。

In [ ]:
def low_level_actions(task_s, hz):
    """低层一共出多少个动作。"""
    return math.floor(task_s * hz)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 两只钟")
no_int = high_level_calls(8, 1.0, [])
with_int = high_level_calls(8, 1.0, [4.3])
print(f"  8 秒任务、每秒一次：高层想 {len(no_int)} 次")
print(f"  第 4.3 秒插话：高层想 {len(with_int)} 次，时刻 {with_int}")
print(f"  低层 50 Hz × 8 秒 = {low_level_actions(8, 50)} 个动作")
g = SYSTEMS["GR00T N1"]
print(f"  GR00T N1：高层 {g['高层 Hz']} Hz、低层 {g['低层 Hz']} Hz，16 步 {g['16 步 chunk 用时 ms']} ms")

## 3. 代码里才看得到的

1. 插话不是"多想一次再接着原节奏"，而是从插话时刻重新计时：所以插话后的几次是 4.3、5.3、6.3、7.3 秒。
2. 低层动作数用 floor：8 × 50 正好整数，换成 7.99 秒就会少一个。

## 4. 自己改一改

- 加第二次插话（比如 6.0 秒），数一数高层想几次。
- 把高层周期改成 0.5 秒，比较想的次数。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 心里想路口，手上调方向：想得慢，做得快。
> 纸条好懂信息少，灌想法信息多看不懂。
> 先有动作再编话，倒推出高层的教材。